In [1]:
import pandas as pd
import numpy as np
import os

# Define paths
RESULTS_CSV = r'inference\inference_results.csv'
METADATA_CSV = r'cleaned_dataset\new_csv\inference_metadata_with_cutoffs.csv'
OUTPUT_FULL_CSV = r'inference\inference_merge.csv'

# 1. Load both dataframes
df_results = pd.read_csv(RESULTS_CSV)
df_meta = pd.read_csv(METADATA_CSV)

# 2. Prepare metadata: Drop the empty length/width/end_pt/curve columns 
# so we can fill them with predictions and NaNs
cols_to_keep = ['dot', 'ngay', 'egg_id', 'side', 'image_path', 'label', 'cutoff_l', 'cutoff_r']
df_meta_clean = df_meta[cols_to_keep]

# 3. Merge metadata with predictions
# We join on image_path to align the right prediction with the right metadata
df_merge = pd.merge(df_meta_clean, df_results, on='image_path', how='left')

# 4. Rename predicted columns to standard names
df_merge = df_merge.rename(columns={
    'pred_length_mm': 'length_mm',
    'pred_width_mm': 'width_mm'
})

# 5. Add/Fill end_pt with NaN as requested
df_merge['end_pt'] = np.nan

# 6. Select and order columns exactly as specified
final_columns = [
    'dot', 'ngay', 'egg_id', 'side', 'image_path', 'label', 
    'length_mm', 'width_mm', 'cutoff_l', 'cutoff_r', 'end_pt'
]

df_full = df_merge[final_columns]

# 7. Save to the new directory
os.makedirs(os.path.dirname(OUTPUT_FULL_CSV), exist_ok=True)
df_full.to_csv(OUTPUT_FULL_CSV, index=False)

print(f"Successfully merged {len(df_full)} records.")
print(f"File saved to: {OUTPUT_FULL_CSV}")

# Display the first few rows to verify
df_full.head()

Successfully merged 4384 records.
File saved to: inference\inference_merge.csv


,dot,ngay,egg_id,side,image_path,label,length_mm,width_mm,cutoff_l,cutoff_r,end_pt
0,1,0,1,A,dot_1_ngay_0_egg_1_side_A.jpg,T,49.06,38.73,"178,427","369,397",NaN
1,1,0,1,B,dot_1_ngay_0_egg_1_side_B.jpg,T,47.67,37.60,"146,406","349,423",NaN
2,1,0,1,C,dot_1_ngay_0_egg_1_side_C.jpg,T,50.98,40.07,"146,399","340,423",NaN
3,1,0,10,A,dot_1_ngay_0_egg_10_side_A.jpg,T,49.41,36.95,"165,421","360,411",NaN
4,1,0,10,B,dot_1_ngay_0_egg_10_side_B.jpg,T,48.78,36.49,"159,418","354,415",NaN


In [2]:
import os
import pandas as pd
import numpy as np
import cv2
from scipy.interpolate import make_interp_spline

# --- Configuration ---
MASK_DIR = r'cleaned_dataset\split\inference_masks'
CSV_INPUT = r'inference\inference_merge.csv'
CSV_OUTPUT = r'cleaned_dataset\new_csv\inference_full.csv'
RECON_OUT_DIR = r'cleaned_dataset\recon\inference'

os.makedirs(RECON_OUT_DIR, exist_ok=True)

def parse_coord(coord_str):
    if pd.isna(coord_str) or coord_str == "":
        return None
    try:
        cleaned = str(coord_str).replace('[', '').replace(']', '').strip()
        return np.array([int(float(c)) for c in cleaned.split(',')])
    except:
        return None

def process_dataset():
    df = pd.read_csv(CSV_INPUT)
    
    for idx, row in df.iterrows():
        img_name = row['image_path']
        img_path = os.path.join(MASK_DIR, img_name)
        if not os.path.exists(img_path):
            continue

        # 1. Load + preprocess
        img = cv2.imread(img_path)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)
        cnt_sq = cnt.squeeze()

        l_cutoff = parse_coord(row['cutoff_l'])
        r_cutoff = parse_coord(row['cutoff_r'])
        if l_cutoff is None or r_cutoff is None:
            continue

        L_mm, W_mm = row['length_mm'], row['width_mm']

        x_min, x_max = np.min(cnt_sq[:, 0]), np.max(cnt_sq[:, 0])
        y_min, y_max = np.min(cnt_sq[:, 1]), np.max(cnt_sq[:, 1])

        x_center = x_min + (x_max - x_min) // 2
        y_bottom_target = y_min + int((x_max - x_min) * (L_mm / W_mm))

        # 2. Split contour
        left_half = cnt_sq[cnt_sq[:, 0] < x_center]
        right_half = cnt_sq[cnt_sq[:, 0] >= x_center]

        if len(left_half) == 0 or len(right_half) == 0:
            continue

        # guide points (slightly above cutoff)
        l_guide = left_half[np.argmin(np.abs(left_half[:, 1] - (l_cutoff[1] - 15)))]
        r_guide = right_half[np.argmin(np.abs(right_half[:, 1] - (r_cutoff[1] - 15)))]

        anchors = np.array([
            l_guide,
            l_cutoff,
            [x_center, y_bottom_target],
            r_cutoff,
            r_guide
        ])

        # 3. Spline (chord-length parameterization)
        distances = np.linalg.norm(np.diff(anchors, axis=0), axis=1)
        u = np.insert(np.cumsum(distances), 0, 0)

        spline_x = make_interp_spline(u, anchors[:, 0], k=3)
        spline_y = make_interp_spline(u, anchors[:, 1], k=3)

        u_fine = np.linspace(u[1], u[3], 100)
        curve_pts = np.column_stack((spline_x(u_fine), spline_y(u_fine))).astype(np.int32)

        # 4. Visualization (WHITE bg, BLACK lines)
        h, w = img.shape[:2]
        canvas = np.ones((h, w, 3), dtype=np.uint8) * 255  # white

        # clipping mask (keep only upper contour)
        if (r_cutoff[0] - l_cutoff[0]) != 0:
            m = (r_cutoff[1] - l_cutoff[1]) / (r_cutoff[0] - l_cutoff[0])
        else:
            m = 0
        b = l_cutoff[1] - m * l_cutoff[0]

        mask = np.zeros((h, w), dtype=np.uint8)
        clip_poly = np.array([
            [0, 0],
            [w, 0],
            [w, int(m * w + b)],
            [0, int(m * 0 + b)]
        ], dtype=np.int32)

        cv2.fillPoly(mask, [clip_poly], 255)

        # draw contour on temp layer
        temp = np.zeros((h, w), dtype=np.uint8)
        cv2.drawContours(temp, [cnt], -1, 255, 1)

        # apply clipping
        upper_only = cv2.bitwise_and(temp, temp, mask=mask)

        # paint onto white canvas
        canvas[upper_only > 0] = [0, 0, 0]

        # draw spline (black)
        cv2.polylines(
            canvas,
            [curve_pts.reshape((-1, 1, 2))],
            isClosed=False,
            color=(0, 0, 0),
            thickness=1
        )

        # 5. Save outputs
        df.at[idx, 'end_pt'] = f"{x_center},{y_bottom_target}"
        df.at[idx, 'curve'] = "|".join([f"{p[0]},{p[1]}" for p in curve_pts])

        cv2.imwrite(os.path.join(RECON_OUT_DIR, img_name), canvas)

    df.to_csv(CSV_OUTPUT, index=False)
    print("Done. White background + black outlines.")

process_dataset()

C:\Users\Dell\AppData\Local\Temp\ipykernel_16096\303738569.py:128: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '255,456' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, 'end_pt'] = f"{x_center},{y_bottom_target}"


Done. White background + black outlines.
